# Resource Store Playground

In [ ]:
import warnings

from pathlib import Path
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
import RESource.RESources as RES_module

from RESource import utility as utils
from RESource.hdf5_handler import DataHandler

import RESource.visual_styles as styles
style_path = Path(styles.__file__).parent / "elsevier.mplstyle"
plt.style.use(style_path)
# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning)

cfg_path='config/CAN_baseline.yaml'
# cfg_path='config/config_CAN_policy1.yaml'
cfg=utils.load_config(cfg_path)

In [ ]:
country_name=cfg.get('country','Canada') 
country_kwd=country_name.replace(' ','')


## Set Required Args to Initialize Objects

- All steps are integrated in this 'RES_module.build()' method.

In [ ]:
# import RESource.RESources as RES

# # Iterate over provinces for both solar and wind resources
# resource_types = ['wind','solar']  # 'solar'
# provinces=['BC']  #,'AB','SK','ON','NS','MB'
# for province_code in provinces:
#     for resource_type in resource_types:
#         required_args = {
#             "config_file_path": 'config/config_CAN_policy1.yaml',
#             "region_short_code": province_code,
#             "resource_type": resource_type
#         }
        
#         # Create an instance of Resources and execute the module
#         RES_module = RES.RESources_builder(**required_args)
        
    
#     # For complete workflow, uncomment the following line
#         RES_module.build(select_top_sites=True,
#                          use_pypsa_buses=False)  


----

In [ ]:
# Construct region_options as a list of tuples: (name, code)
region_options = [(cfg['region_mapping'][code]['name'], code) for code in cfg['region_mapping']]
region_code = 'BC'  # Default selection, change as needed

In [ ]:
# resource_type_dropdown = widgets.Dropdown(
#     options=['wind', 'solar'],
#     value='solar',
#     description='Resource:',)
# display(resource_type_dropdown)


In [ ]:
# resource_type = resource_type_dropdown.value


# required_args = {
#     "config_file_path": cfg_path,
#     "region_short_code": region_code,
#     "resource_type": resource_type
# }


# # Create an instance of Resources and execute the module
# Builder = RES_module.RESources_builder(**required_args)

# [Exploratory]

##  Define run/scenario

In [ ]:
RUN_ID= cfg.get('Scenario').get('run_id')+"_20260522"
store=f"data/store/{country_kwd}/{RUN_ID}/resources_{country_kwd}_{region_code}_{RUN_ID}.h5"# f"../data/store/resources_{province_code}.h5" 
res_store=DataHandler(store,show_structure=True) # the DataHandler object could be initiated without the store definition as well.

In [ ]:
cells=res_store.from_store('cells')
boundary=res_store.from_store('boundary')

### Step 7: Clusterized Representation of the Sites

- Groups grid cells into clusters based on spatial or resource characteristics to enable aggregated analysis.
- Produces time series data for each cluster, summarizing the resource and capacity factor information at the cluster leve

In [ ]:
for resource_type in ['wind', 'solar']:

    required_args = {
        "config_file_path": cfg_path,
        "region_short_code": region_code,
        "resource_type": resource_type
    }

    # Create an instance of Resources and execute the module
    Builder = RES_module.RESources_builder(**required_args)
    Builder
    Clusters=Builder.get_clusters(scored_cells=cells)
    ClusterTS=Builder.get_cluster_timeseries()


In [ ]:
solar_clusters=res_store.from_store('clusters/solar')
wind_clusters=res_store.from_store('clusters/wind')
solar_clusters_ts=res_store.from_store('timeseries/clusters/solar')
wind_clusters_ts=res_store.from_store('timeseries/clusters/wind')
dissolved_indices_solar=res_store.from_store('dissolved_indices/solar')
dissolved_indices_wind=res_store.from_store('dissolved_indices/wind')

---

- Interactive Map

In [ ]:
# from RESource import visuals as vis
# vis.make_lcoe_map(
#     wind_gdf=cells,
#     solar_gdf=cells,
#     sub_national_unit_tag=Builder.sub_national_unit_tag,
#     save_path=f"CAN_lcoe_map_{RUN_ID}.html",
#     basemap_tiles="Esri WorldGrayCanvas",
#     wind_lcoe_max=150,
#     solar_lcoe_max=65,
# )

In [ ]:
# wind_clusters[wind_clusters['lcoe']<=100].explore('potential_capacity')

# Playground for Top Site Selection

In [ ]:
resource_clusters_solar,cluster_timeseries_solar=RES_module.select_top_sites(solar_clusters,
                                                                solar_clusters_ts,
                                                                    resource_max_capacity=10)

resource_clusters_wind,cluster_timeseries_wind=RES_module.select_top_sites(wind_clusters,
                                                                wind_clusters_ts,
                                                                    resource_max_capacity=20)

In [ ]:
for resource_type in ['solar', 'wind']:
    RES_module.export_results(resource_type,
                              region_code,
                              resource_clusters_solar if resource_type=='solar' else resource_clusters_wind,
                              cluster_timeseries_solar if resource_type=='solar' else cluster_timeseries_wind,
                              save_to=f'results/Canada/BC/{RUN_ID}')